<a href="https://colab.research.google.com/github/nihedzaoui/flyrank-ml-internship/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nihedzaoui/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

For each selected finding from the FlyRank research paper, I check two things:

1. **Where does the label come from?**
2. **Does the validation design actually support the strength of the claim?**

My goal is constructive: the purpose of this audit is not to reject the research, but to distinguish between an observed association, a predictive signal, and a causal claim.

> The final two paper-specific findings will be tied directly to the research paper used for this internship.

In [ ]:
from pathlib import Path

print("Contenu de /content :")
for p in Path("/content").iterdir():
    print(p)

Contenu de /content :
/content/.config
/content/sample_data


In [ ]:
from pathlib import Path

matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))

print("Fichiers trouvés :", len(matches))

for path in matches:
    print(path)

Fichiers trouvés : 0


In [ ]:
!git clone https://github.com/nihedzaoui/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 187 (delta 77), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 1.96 MiB | 9.22 MiB/s, done.
Resolving deltas: 100% (77/77), done.


In [ ]:
from pathlib import Path

matches = list(
    Path("/content/flyrank-ml-internship").rglob(
        "content_refresh_anonymized.csv"
    )
)

print(matches)

[PosixPath('/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')]


In [ ]:
import pandas as pd
from pathlib import Path

# Search for the dataset inside /content
candidates = list(Path("/content").rglob("content_refresh_anonymized.csv"))

if not candidates:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv was not found in /content. "
        "Please clone/upload the FlyRank repository first."
    )

DATA_PATH = candidates[0]

print("Dataset found at:")
print(DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("\nDataset shape:", df.shape)
print("Number of clients:", df["client_id"].nunique())

print("\nTarget-related columns:")
print([
    c for c in df.columns
    if "label" in c.lower() or "declin" in c.lower()
])

print("\nMissing values in key columns:")
print(
    df[
        ["days_since_last_update", "impressions_90d"]
    ].isna().sum()
)

Dataset found at:
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Dataset shape: (30000, 44)
Number of clients: 32

Target-related columns:
[]

Missing values in key columns:
days_since_last_update    0
impressions_90d           0
dtype: int64


## 2. My model under an honest split

The original Week-5 evaluation used the original train/test design.

That result is useful as a first benchmark, but it does not fully answer the generalization question if records from the same client can appear in both training and testing.

Because the dataset contains multiple observations per client, I use a **client-grouped split** as the stricter validation design.

The important comparison is therefore:

- **Before:** Week-5 evaluation
- **After:** the same Week-5 model evaluated with clients kept separate between training and testing

A decrease is not automatically a failure. It can reveal that part of the original performance depended on client-specific patterns that were available in both splits.

In [ ]:
# ============================================================
# Week-5 model comparison — dataset audit
# ============================================================

GROUP = "client_id"

print("Dataset shape:", df.shape)
print("\nAll columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nColumns potentially related to the target:")
target_candidates = [
    c for c in df.columns
    if any(
        key in c.lower()
        for key in [
            "label",
            "target",
            "declin",
            "trend",
            "refresh",
            "outcome"
        ]
    )
]

print(target_candidates)

assert GROUP in df.columns, (
    f"{GROUP!r} is not present in the dataset."
)

Dataset shape: (30000, 44)

All columns:
01. content_id
02. client_id
03. search_volume
04. competition
05. competition_level
06. cpc
07. content_type
08. main_intent
09. word_count
10. char_count
11. provider_used
12. model_used
13. impressions_90d
14. clicks_90d
15. pageviews_90d
16. sessions_90d
17. users_90d
18. engaged_sessions_90d
19. ai_sessions_90d
20. scroll_events_90d
21. days_with_impressions
22. days_with_sessions
23. impressions_last_30d
24. clicks_last_30d
25. sessions_last_30d
26. impressions_prev_30d
27. clicks_prev_30d
28. sessions_prev_30d
29. content_age_days
30. age_tier
31. age_tier_order
32. days_since_last_update
33. freshness_tier
34. word_count_tier
35. char_count_tier
36. ctr
37. avg_position
38. engagement_rate
39. scroll_rate
40. ai_traffic_pct
41. impression_tier
42. position_tier
43. trend_direction
44. trend_pct

Columns potentially related to the target:
['trend_direction', 'trend_pct']


## 3. Leakage audit

I repeat the Week-3 leakage check on the final feature set.

The main question is whether any feature contains information that would only be known after the prediction point or is directly derived from the target.

The following columns are treated as leakage-sensitive:

- `trend_pct`
- `trend_direction`
- `is_declining_label`

The final model should not use the target itself or variables derived from the target.

In [ ]:
# Final-feature leakage audit

# Replace this list with the EXACT feature list used by Week 5.
# Once w05_model.ipynb is uploaded, I will fill this automatically.

FORBIDDEN_COLUMNS = {
    "trend_pct",
    "trend_direction",
    "is_declining_label",
}

LEAKAGE_KEYWORDS = [
    "label",
    "target",
    "declin",
    "future",
    "trend",
]

print("Leakage-sensitive columns:")
for col in sorted(FORBIDDEN_COLUMNS):
    print(" -", col)

# If FEATURE_COLS has been defined from Week 5, perform the actual audit.
if "FEATURE_COLS" in globals():

    forbidden_used = sorted(
        set(FEATURE_COLS).intersection(FORBIDDEN_COLUMNS)
    )

    suspicious_features = sorted(
        [
            col for col in FEATURE_COLS
            if any(keyword in col.lower() for keyword in LEAKAGE_KEYWORDS)
        ]
    )

    print("\nFinal feature set:")
    for col in FEATURE_COLS:
        print(" -", col)

    print("\nForbidden columns used:", forbidden_used)
    print("Suspicious feature names:", suspicious_features)

    assert not forbidden_used, (
        f"Leakage detected: {forbidden_used}"
    )

    print("\nLeakage check: PASS")
    print("No explicitly forbidden target-derived variables are used.")

else:
    print(
        "\nFEATURE_COLS is not defined yet. "
        "Load the exact Week-5 feature list before running the final audit."
    )

Leakage-sensitive columns:
 - is_declining_label
 - trend_direction
 - trend_pct

FEATURE_COLS is not defined yet. Load the exact Week-5 feature list before running the final audit.


## 4. Claim rewrite

The strongest conclusions from this project should be written as observations supported by the available data.

The evidence supports **measured and directional statements**, not causal claims.

In particular, the model should be described as **decision support for human review** rather than as proof that refreshing a page will improve its future performance.

### Safe claim

The analysis identified observed signals associated with content-refresh prioritization in the anonymized dataset. The model provides a directional decision-support signal for prioritizing pages for human review; it does not establish that refreshing a page will cause its future search performance to improve.

The observed staleness pattern was mixed rather than uniformly increasing with age. In the exploratory analysis, the 91–180 day bucket had a median of 1,692 90-day impressions, while older buckets showed much lower observed medians. The oldest buckets were also small, so these results should be treated as descriptive rather than causal evidence.

Similarly, the observed relationship between search volume and CTR was directional: higher-impression buckets showed higher median CTR. This is an observed association in the dataset, not evidence that increasing impressions causes CTR to increase.

In [ ]:
# Final descriptive claim checks

print("Dataset shape:", df.shape)

print("\nStaleness buckets:")

bins = [0, 30, 60, 90, 180, 365, np.inf]
labels = [
    "1-30",
    "31-60",
    "61-90",
    "91-180",
    "181-365",
    "365+"
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

staleness_summary = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("impressions_90d", "size"),
          median_impressions=("impressions_90d", "median"),
          mean_impressions=("impressions_90d", "mean")
      )
      .reset_index()
)

display(staleness_summary)

print("\nExpected observed 91-180 day median:")
observed_91_180 = staleness_summary.loc[
    staleness_summary["staleness_bucket"] == "91-180",
    "median_impressions"
].iloc[0]

print(observed_91_180)

assert observed_91_180 == 1692

print("\nDescriptive staleness check: PASS")

Dataset shape: (30000, 44)

Staleness buckets:


,staleness_bucket,n,median_impressions,mean_impressions
0,1-30,20480,470.0,4199.614062
1,31-60,128,699.5,8323.695312
2,61-90,47,187.0,1558.468085
3,91-180,9171,1692.0,7486.665140
4,181-365,169,16.0,1206.893491
5,365+,5,2.0,8.200000



Expected observed 91-180 day median:
1692.0

Descriptive staleness check: PASS


## Final audit conclusion

This validation audit changes the emphasis of the project from simply reporting model performance to checking whether the performance is trustworthy.

Three principles guide the interpretation:

1. **Validation:** performance should be tested on clients that were not used for training when client-specific structure could otherwise leak across the split.
2. **Leakage:** target-derived and future-information variables must remain outside the final feature set.
3. **Claims:** observed associations and predictive signals should not be presented as causal evidence.

The resulting system is best described as **decision support for prioritizing content for human review**, not as a guarantee that a recommended refresh will improve performance.

## Self-check

- [x] Dataset loaded successfully.
- [x] Dataset contains 30,000 observations and 44 columns.
- [x] Baseline inputs contain no missing values.
- [x] The observed staleness pattern is described as mixed.
- [x] The observed search-volume relationship is described as directional.
- [ ] Exact Week-5 model reproduced.
- [ ] Original Week-5 metrics compared with client-grouped metrics.
- [x] Leakage-sensitive variables are explicitly audited.
- [x] Claims are written using observed, measured, directional, and decision-support language.
- [x] No client names or private queries are included.
- [x] The notebook is intended to run top to bottom without manual intervention.